In [110]:
import os
import pandas as pd
import numpy as np
import datetime

# Load Data

In [111]:
downtime_raw = pd.read_csv(os.path.join("data", "downtime.csv"))
production_raw = pd.read_csv(os.path.join("data", "merged.csv"))

# Inspect Data

In [112]:
downtime_df = pd.DataFrame()
production_df = pd.DataFrame()

# Format as Datetime

In [113]:
downtime_df["Timestamp"] = pd.to_datetime(downtime_raw["Production Date"] + " " + downtime_raw["Time"], dayfirst=True)
downtime_df["Date"] = downtime_df["Timestamp"].dt.date
production_raw["Set Time"] = pd.to_datetime(production_raw["Set Time"], dayfirst=True)
production_raw["Date"] = production_raw["Set Time"].dt.date

# Get relevant production processes

In [114]:
downtime_unique_dates = pd.DataFrame(downtime_df["Timestamp"].dt.date.unique(), columns=["Date"])

In [115]:
# Get production processes that contains downtime dates
production_datetime_filtered = production_raw[production_raw["Date"].isin(downtime_df["Date"])].copy()
# Keep only downtime rows that is in range of production time
downtime_df = downtime_df[downtime_df["Date"].isin(production_datetime_filtered["Date"])]

# Keeping columns that has downtime

In [163]:
production_pos = production_datetime_filtered["Set Time"].searchsorted(downtime_df["Timestamp"], side="right") - 1

In [166]:
matches = production_datetime_filtered.iloc[production_pos.astype(int)]

# Preprocess data

## Drop duplicates and unnecessary columns

In [171]:
matches.drop_duplicates(inplace=True)

C:\Users\veril\AppData\Local\Temp\ipykernel_7264\3956366675.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches.drop_duplicates(inplace=True)


In [173]:
matches.drop(["Set Time", "VYP batch", "Date"], axis=1, inplace=True)

C:\Users\veril\AppData\Local\Temp\ipykernel_7264\1564903515.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches.drop(["Set Time", "VYP batch", "Date"], axis=1, inplace=True)


In [ ]:
## Inspecting unique values

In [176]:
columns_to_be_converted = []
for col in matches.columns:
    if "SP" in col:
        continue
    unique_values = len(matches[col].unique())
    print(f"{col} Has unique values: {unique_values}")
    if unique_values > 1 and unique_values < 10 and col != "batch":
        columns_to_be_converted.append(col)

Part Has unique values: 3
Extract tank Level Has unique values: 22
FFTE Discharge density Has unique values: 12
FFTE Discharge solids Has unique values: 19
FFTE Feed flow rate PV Has unique values: 20
FFTE Feed solids PV Has unique values: 20
FFTE Heat temperature 1 Has unique values: 19
FFTE Heat temperature 2 Has unique values: 20
FFTE Heat temperature 3 Has unique values: 20
FFTE Production solids PV Has unique values: 21
FFTE Steam pressure PV Has unique values: 14
TFE Input flow PV Has unique values: 20
TFE Level Has unique values: 22
TFE Motor current Has unique values: 20
TFE Motor speed Has unique values: 2
TFE Out flow PV Has unique values: 20
TFE Product out temperature Has unique values: 1
TFE Production solids PV Has unique values: 21
TFE Production solids density Has unique values: 13
TFE Steam pressure PV Has unique values: 11
TFE Steam temperature Has unique values: 22
TFE Tank level Has unique values: 22
TFE Temperature Has unique values: 14
TFE Vacuum pressure PV Has u

In [178]:
matches.drop(["TFE Product out temperature"], axis=1, inplace=True)

C:\Users\veril\AppData\Local\Temp\ipykernel_7264\4039993084.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matches.drop(["TFE Product out temperature"], axis=1, inplace=True)


In [181]:
matches.to_csv("data/processes_with_downtime.csv", index=False)